# Seer Quality Control

**This pipeline performs the quality control (QC) on:** 
- each sample 
- each protein 

Samples with valid values in less than 20% (default cut-off) of all proteins measured and proteins with more than 10% (default cut-off) of excluded values are removed. Cuto-ffs can be customised by the user.

The only argument that the user has to feed is the path where the raw Seer table is located (.tsv file). The pipeline returns a list with two data frames:
- one table including protein info (Protein_info)
- one table including protein groups values (Protein_values)

**Some important notes:**
- it works only with the raw Seer .tsv file sent by Seer (Panel data)
- the pipeline does not remove outliers

## Helper functions

In [ ]:
# Sample Quality Control function
# This function calculates how many missing values a sample has (in percentage)
# @param data A data frame
sample_QC <- function(data) {
    
    sample_miss <- data.frame(Sample = c(),
                              Row_index = c(),
                              Missingness_pct = c())
    
    for(i in 1:nrow(data)) {
        
        sample_miss[i, "Sample"] <- data$Sample_Name[i]
        sample_miss[i, "Row_index"] <- i
        sample_miss[i, "Missingness_pct"] <- round(sum(is.na(data[i, 2:ncol(data)]))/
                                                   ncol(data[i, 2:ncol(data)]),
                                                   2)
    }
    
    return(sample_miss)
}

In [ ]:
# Protein Quality Control function
# This function calculates how many missing values a protein has (in percentage)
# @param data A data frame
protein_QC <- function(data) {
    
    prot_miss <- data.frame(Protein = c(),
                            Col_index = c(),
                            Missingness_pct = c())
    
    for(i in 2:ncol(data)) {
        
        prot_miss[i - 1, "Protein"] <- colnames(data)[i]
        prot_miss[i - 1, "Col_index"] <- i
        prot_miss[i - 1, "Missingness_pct"] <- round(sum(is.na(data[, i]))/
                                                       nrow(data),
                                                       2)
              
    }
    
    return(prot_miss)
    
}

## Pipeline

In [ ]:
# Seer_Panel_QC function
# @param path Location of Olink Explore file written as path
# @param cutoff_sample cutoff to exclude samples based on missingness percentage. Default value is 0.8
# @param cutoff_protein cutoff to exclude proteins based on missingness percentage. Default value is 0.1
Seer_Panel_QC <- function(path, cutoff_sample = 0.8, cutoff_protein = 0.1) {
    
    start_time <- Sys.time()
    
    # Libraries installing/loading
    required_packages <- "data.table"    
    packages_to_install <- required_packages[!required_packages %in% installed.packages()]
    if(length(required_packages)) {
        install.packages(packages_to_install)
    }
    sapply(required_packages, require, character = TRUE)    
    
    # Data loading
    data_long <- fread(path)
    message("The table in long format has", nrow(data_long), "rows and", ncol(data_long), "columns")
    
    # Data formatting
    ## Step 1: replace blanks and parenthesis in the column names for underscores
    colnames(data_long) <- gsub(" ", "_", colnames(data_long))
    colnames(data_long) <- gsub("(", "", colnames(data_long), fixed = TRUE)
    colnames(data_long) <- gsub(")", "", colnames(data_long), fixed = TRUE)
    ## Step 2: in key data, replace semicolons for underscores and hiphens for full stops
    data_long$Protein_Group <- gsub(";", "_", data_long$Protein_Group, fixed = TRUE)
    data_long$Protein_Group <- gsub("-", ".", data_long$Protein_Group, fixed = TRUE)
    data_long$Gene_Names <- gsub(";", "_", data_long$Gene_Names, fixed = TRUE)
    
    # Table to match Protein_Group, Protein_Names, Gene_Names, Biological_Process, Molecular_Function, and Cellular_Component
    ## Step 1: select the cols
    cols_prot <- c("Protein_Group", "Protein_Names", "Gene_Names", 
                   "Biological_Process", "Molecular_Function", "Cellular_Component")
    ## Step 2: subset the cols
    data_long_subset <- data_long[, ..cols_prot]
    ## Step 3: extract unique Protein_Group values
    prot_group_name_gene <- unique(data_long_subset, by = "Protein_Group")
    prot_group_name_gene_count <- nrow(prot_group_name_gene)
    ## Step 4: count unique genes
    unique_genes_count <- nrow(unique(data_long_subset, by = "Gene_Names"))
    ## Data table to data frame
    prot_group_name_gene <- setDF(prot_group_name_gene)
    message("A table containing protein groups, protein names, gene names, and GO data has been generated. The table contains", 
            prot_group_name_gene_count, "unique protein groups and", unique_genes_count, "unique genes")
    
    message("Another table with protein group values will be generated")    
    # Data needs to be reshaped from long to wide format
    ## Since isoforms have the same gene name, I will use Protein_Group as colnames because it is the only unique value for each protein
    data_wide <- dcast(data = data_long[, .(Sample_Name, Protein_Group, Normalized_Intensity_Log10)], # subset to reshape to wide
                       formula = Sample_Name ~ Protein_Group, # variable that will stay as a column ~ variable that will go to column names
                       value.var = "Normalized_Intensity_Log10") # values that will fill the new columns
    message("After reshaping, the table in wide format has", 
            nrow(data_wide), "rows and", ncol(data_wide), "columns")    
    ## Data table to data frame
    data_wide <- setDF(data_wide)
    
    # Sample QC: samples with more than 80% missingness will be excluded
    ## Step 1: sample missingness calculation
    sample_missingness <- sample_QC(data_wide)
    bad_samples_index <- subset(sample_missingness, Missingness_pct > cutoff_sample)$Row_index
    bad_samples_number <- length(bad_samples_index)
    message("There is/are", bad_samples_number, "sample/s with over", cutoff_sample*100, "% missingness")
    ## Step 2: remove bad samples (if they exist)
    if(bad_samples_number) {
       data_wide <- data_wide[-bad_samples_index,] 
       message("After sample QC, the table in wide format has", 
               nrow(data_wide), "rows and", ncol(data_wide), "columns")
    }   
    
    # Protein QC: proteins with more than 10% missingness will be excluded
    ## Step 1: protein missingness calculation
    protein_missingness <- protein_QC(data_wide)
    bad_proteins_index <- subset(protein_missingness, Missingness_pct > cutoff_protein)$Col_index
    bad_proteins_number <- length(bad_proteins_index)
    message("There is/are", bad_proteins_number, "protein/s with over", cutoff_protein*100, "% missingness")
    ## Step 2: remove bad proteins (if they exist)
    if(bad_proteins_number) {
       data_wide <- data_wide[-bad_proteins_index]
       message("After protein QC, the table in wide format has", 
               nrow(data_wide), "rows and", ncol(data_wide), "columns")
    }
    
    message("Both tables are returned as a list: first table is Protein_info and second table is Protein_values")
    
    # Execution time calculation
    end_time <- Sys.time()
    
    time_diff <- end_time - start_time
    
    message("Execution time:", round(time_diff, 2), attr(time_diff, "units")) 
    
    return(list(Protein_info = prot_group_name_gene,
                Protein_values = data_wide))   
    
}

In [ ]:
## run your pipeline below:
your_qced_file <- Seer_Panel_QC("your_path")
## save your tables
### Protein info table
write.table(your_qced_file$Protein_info,
            "your_path",
            sep = "\t",
            row.names = FALSE,
            quote = FALSE)
### Protein data (values) table
write.table(your_qced_file$Protein_values,
            "your_path",
            sep = "\t",
            row.names = FALSE,
            quote = FALSE)